<a href="https://colab.research.google.com/github/rohit-backend-dev/Agentic-AI/blob/main/1_Build_a_Simple_AI_Agent_With_OpenAI_Agents_SDK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TASK 1. SETTING UP YOUR ENVIRONMENT


## Before we can build our first AI application, we need two things:

1. The **Google Gen AI SDK** (`google-genai`) installed.
2. Your **Gemini API key** from Google AI Studio to communicate with Gemini models.

### Security Tip 📌

Store your API key securely. If you're using a local environment, create a file named **`.env`** in the same directory as this notebook and add:

```text
GEMINI_API_KEY=AIzaSyYourSecretGeminiAPIKeyHereXXXXXXXX
```

If you're using **Google Colab**, it's recommended to store your key in **Colab Secrets**:

* Open the **🔑 Secrets** tab in the left sidebar.
* Create a secret named **`GEMINI_API_KEY`**.
* Paste your Gemini API key as the value.
* Enable **Notebook access**.

Now let's install the required package and configure the Gemini client.

### Install the Gemini SDK

```python
!pip install -q google-genai
```



- **You can access the OpenAI API here: https://openai.com/api/**

In [ ]:
# Install the specific version of openai-agents
!pip install -q google-genai

In [ ]:
# Install required Python packages:
# langchain-openai==0.2.1: a specific version of LangChain’s OpenAI integration library
# LangChain is a framework that makes it easier to build applications powered by large language models
# It provides tools to connect to AI models, store context, and chain multiple steps together.
!pip install python-dotenv langchain-openai==0.2.1

# Upgrade the Pydantic library (used for validating and organizing data)
# This ensures compatibility with the installed packages above
!pip install --upgrade pydantic

In [ ]:
# Import required modules
import os
from google import genai
from IPython.display import display, Markdown
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Get the Gemini API key from the environment
gemini_api_key = os.getenv("GEMINI_API_KEY")

# Configure the Gemini client
gemini_client = genai.Client(api_key=gemini_api_key)

print("Gemini client successfully configured.")

# View the first few characters of the API key
print(gemini_api_key[:5])

OpenAI client successfully configured.
sk-pr


In [ ]:
# A Function used to Show the given text using Markdown formatting in a Jupyter notebook
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

# TASK 2. BUILD & RUN YOUR FIRST AI AGENT USING google-genai AGENT SDK

It's time to build our first AI application with Gemini!

Unlike the OpenAI Agents SDK, the Gemini SDK does not have an Agent class. Instead, you create a Gemini client and provide the agent's behavior through a prompt (instructions) each time you call the model.

The key components are:

client – The Gemini client we configured earlier. It connects your application to Google's Gemini models.
instructions – The system instructions that define the AI's role, personality, and behavior (for example, "You are a Fact Checker").
model – The Gemini model used to generate responses, such as:
models/gemini-3.6-flash – Fast, efficient, and recommended for most tasks.
models/gemini-3.5-flash – Fast and lightweight.
models/gemini-2.5-pro – Better for complex reasoning and coding tasks.
contents – The user's input or question that the model should respond to.

In [8]:
from google.colab import userdata
from google import genai
from IPython.display import display, Markdown

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

statement = "The Great Wall of China is visible from space with the naked eye."

response = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents=f"""
You are a fact checker.

Verify the following statement.
Explain whether it is true or false with a short reason.

Statement:
{statement}
"""
)

display(Markdown("### 🤖 Gemini Response"))
display(Markdown(response.text))

### 🤖 Gemini Response

**Verdict:** False

**Reason:** 
This is a well-known myth. The Great Wall of China is only about 5 to 9 meters (15 to 30 feet) wide and is constructed from materials that blend in with the surrounding terrain. Astronauts—including China’s own Yang Liwei—and NASA have repeatedly confirmed that it cannot be seen from Low Earth Orbit (or the Moon) with the unaided human eye without magnification or special camera lenses.

Let's put our `fact_checker_agent` to work! We'll use the `Runner.run()` method to send it a statement to fact-check. The agent will analyze the statement and return its verdict along with a brief explanation.

**PRACTICE OPPORTUNITY:**
- **Now it's your turn to experiment with OpenAI API Agents SDK; perform the following tasks:**
   - **Change the text inside the `statement` variable. Try a different fact, like `"The tallest mountain in the world is Mount Everst"` See how the AI agent responds!**
   - **Try a different AI model, change the model from `model="gpt-4o-mini"` to `model="gpt-5-mini"`**

## TASK 3. CHECK IF THE AI AGENT CAN RECALL INFORMATION (NO MEMORY) & CHECK TRACES

1. Check if Gemini remembers information (No Memory)

Each generate_content() call is independent unless you send previous conversation history.

In [18]:
response1 = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents="My favorite programming language is Python."
)

print(response1.text)

Python is a great choice! It's widely loved for its clean, readable syntax and incredible versatility. 

What do you mostly use it for? Are you working on web development, data science, automation, AI, or just building cool projects for fun?


Now ask a follow-up in a new request:

In [19]:
response2 = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents="What is my favorite programming language?"
)

print(response2.text)

I don't have access to your personal preferences or memory from past conversations, so I don't know! 

What is your favorite programming language? (Python, JavaScript, Rust, C++, or something else?)


Show how to provide memory manually

In [22]:
conversation = """
User: My favorite programming language is Python.
Assistant: That's great!

User: What is my favorite programming language?
"""

response = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents=conversation
)

response2 = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents="What is my favorite programming language?"
)

print(response2.text)

I don't know your favorite programming language yet! Since I don't have access to your personal information or past conversations, you haven't told me. 

Is it **Python**, **JavaScript**, **Rust**, **C++**, **Go**, or something else? Give me a few hints, and I can try to guess!


In [13]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="models/gemini-flash-latest",
    contents="Say Hello"
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='Hello! How can I help you today?',
        thought_signature=b'\x12\xab\x04\n\xa8\x04\x01\x11M2\x0f\xe3\x92\n\x08\x97\x15]\xf6*\xd2P\xb1\xec\x19\x01\x9aq8R\xf4q80\xd8)\xa4\xe0\xc7\x95b\xb1\xf0\x83\xf3\xf0\xce\x1b\x181\x89\xc8\x14\xf7X\xa1\x97\xaaW\xdd#\x95\x93\x82\x17\x8b$\xbc\xdb\x0c}8d\x8b\xa6"\x1a\x19V7Eyo\x9c\xdc\xb2\x0e%\xda\xf4q\xf2w\xf6N\xa4b...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.6-flash' prompt_feedback=None response_id='4XliaqbtJuvAqtsP9fC5qA4' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=9,
  prompt_token_count=3,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3
    ),
  ],
  thoughts_token_count=133,
  total_token_count=145
) model_status=No

In [ ]:
check_recall_statement = "What did we discuss in the last message?"

response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)


🤖 Agent's Response:


❌ FALSE: You asked for a fact-check on a statement, but the last message did not contain any discussion or statements to fact-check.

**PRACTICE OPPORTUNITY:**
- **Create a new AI agent named `Tweet Bot` that writes a creative, engaging tweet (≤280 characters) about a given topic using OpenAI Agents SDK. The tweet should include at least one emoji and a hashtag.**
    - **1. Set Up Instructions: Define `Tweet Bot`’s purpose and personality, and configure it to use the latest `gpt-5-mini` model**.  
    - **2. Choose a Topic: For example use "The future of renewable energy"**.  
    - **3. Run the Agent: Use `Runner.run()` to execute the agent and print the generated tweet. Use traces to track the Agent behaviour and list the number of input and output tokens used.**

In [ ]:
print("hello")

hello


# PRACTICE OPPORTUNITY SOLUTIONS

**PRACTICE OPPORTUNITY SOLUTION:**
- **Now it's your turn to experiment with OpenAI API Agents SDK; perform the following tasks:**
   - **Change the text inside the `statement` variable. Try a different fact, like `"The tallest mountain in the world is Mount Everst"` See how the AI agent responds!**
   - **Try a different AI model, change the model from `model="gpt-4o-mini"` to `model="gpt-5-mini"`**

In [14]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

fact_checker_instructions = """
You are a professional fact checker.

Your job is to verify statements.
If a statement is false, explain why.
If it is true, briefly explain why.
"""

statement = "The Great Wall of China is visible from space with the naked eye."

response = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents=f"""
{fact_checker_instructions}

Statement:
{statement}
"""
)

print(response.text)

**Verdict: FALSE**

**Explanation:**
The claim that the Great Wall of China is visible from space with the naked eye is a well-known urban legend. 

Even from Low Earth Orbit (LEO)—the lowest altitude where space begins (about 100 to 200 miles above Earth)—the wall is virtually impossible to see without aid. While the wall is very long, it is only about 15 to 30 feet wide and constructed from local stone, earth, and brick that naturally blend in with the surrounding terrain. To the naked eye, resolving an object that narrow from space is equivalent to trying to spot a human hair from several miles away.

Numerous astronauts have confirmed this fact:
* **Yang Liwei**, China's first astronaut, stated after his 2003 mission that he was unable to see the Great Wall.
* **NASA astronauts**, including Neil Armstrong and Leroy Chiao, have repeatedly clarified that the wall cannot be identified with the naked eye from orbit. 

High-powered camera lenses, telescopic equipment, or specific radar 

**PRACTICE OPPORTUNITY SOLUTION:**
- **Create a new AI agent named `Tweet Bot` that writes a creative, engaging tweet (≤280 characters) about a given topic using OpenAI Agents SDK. The tweet should include at least one emoji and a hashtag.**
    - **1. Set Up Instructions: Define `Tweet Bot`’s purpose and personality, and configure it to use the latest `gpt-5-mini` model**.  
    - **2. Choose a Topic: For example use "The future of renewable energy"**.  
    - **3. Run the Agent: Use `Runner.run()` to execute the agent and print the generated tweet. Use traces to track the Agent behaviour and list the number of input and output tokens used.**

In [ ]:
tweet_bot_instructions = """
Instruction:
Write a short, engaging tweet (≤280 characters) about the given topic. The tweet must include at least one relevant emoji and one relevant hashtag.

Context:
You are **TweetBot**, a witty and creative social-media expert known for crafting viral tweets.

Input:
A topic provided by the user.

Output:
A single tweet (≤280 characters) about the topic, containing at least one relevant emoji and one relevant hashtag.
"""


In [ ]:
#In the Gemini SDK, you can reduce output tokens using the config parameter with max_output_tokens

In [15]:
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents="Write a short tweet about AI.",
    config=types.GenerateContentConfig(
        max_output_tokens=30,   # Very low output limit
        temperature=0.7
    )
)

print(response.text)

AI


In [17]:
from google.colab import userdata
from google import genai

# Configure Gemini client
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

tweet_bot_instructions = """
You are TweetBot, a witty and creative social media expert.before giving response
say to users Hey there! I am tweet Bot

Your task is to write a single tweet (maximum 280 characters)
about the topic provided by the user.

Rules:
- Include at least one relevant emoji.
- Include at least one relevant hashtag.
- Return only the tweet.
"""

topic = "The future of renewable energy"

response = client.models.generate_content(
    model="models/gemini-3.6-flash",
    contents=f"""
{tweet_bot_instructions}

Topic:
{topic}
"""
)

print("🐦 Generated Tweet:\n")
print(response.text)

🐦 Generated Tweet:

Hey there! I am tweet Bot

The future of energy is so bright, even solar panels need sunglasses! 🕶️☀️ From wind turbines spinning up a revolution to smart grids taking over, fossil fuels are finally becoming real fossils. Time to plug into a cleaner tomorrow! 🌿⚡ #RenewableEnergy #CleanTech
